![](Images/banner_health_analytics.png){fig-align="center" width="100%"}

### Health Analytics with Python · Module 02: EDA in Healthcare
***
**This capstone integrates NB-01 through NB-07 into a single reproducible EDA pipeline.**

**Deliverables**
1. Automated data quality report (ydata-profiling)
2. Table 1 — baseline characteristics by readmission status
3. Time-trend of readmissions 2019–2023
4. Comorbidity co-occurrence analysis
5. Missingness audit and imputation plan
6. Geospatial hotspot summary
7. Multi-panel EDA summary figure
8. Written analytic narrative

**Estimated time:** 4 hours | **Level:** Advanced | **Libraries:** all MOD-02 libraries


## 0. Analysis Plan

**Research question:** What patient, clinical, and area-level factors are associated with 30-day hospital readmission?

**Analytic steps:**
1. Load and profile data → flag quality issues
2. Describe cohort (Table 1) stratified by readmission status
3. Explore temporal trends in readmission rates
4. Map comorbidity burden and co-occurrence patterns
5. Characterise and document missingness; choose imputation
6. Map geographic clustering of readmission risk
7. Synthesise findings in a multi-panel figure
8. Draft analytic narrative for a methods section

> **Reproducibility:** Set a global random seed; document all decisions in markdown cells.

## 1. Setup & Data Assembly

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy.stats import chi2_contingency, mannwhitneyu, spearmanr, skew
from scipy.cluster import hierarchy
import warnings
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.02)
plt.rcParams.update({'figure.dpi':120,'savefig.dpi':300,
                     'axes.spines.top':False,'axes.spines.right':False,
                     'figure.facecolor':'white'})

# ── Full capstone dataset ──────────────────────────────────────────────────────
N = 1000
ages      = np.random.normal(62,16,N).clip(18,95).astype(int)
sexes     = np.random.choice(['M','F'],N,p=[0.48,0.52])
payers    = np.random.choice(['Medicare','Medicaid','Private','Self-pay','Other'],N,p=[0.40,0.22,0.28,0.07,0.03])
drg_codes = np.random.choice([470,291,392,690,871,193,247,603],N)
drg_labels= {470:'Major joint replacement',291:'Heart failure & shock',
             392:'Esophagitis/misc GI',690:'Kidney/UTI',871:'Septicemia',
             193:'Simple pneumonia',247:'Perc cardiovasc w stent',603:'Cellulitis'}
los_days  = np.random.gamma(2.5,1.8,N).clip(1,30).astype(int)
charges   = (los_days*np.random.uniform(1800,4200,N)).round(2)
np.random.seed(99)
base = np.random.normal(size=(N,3))
def logistic(x): return 1/(1+np.exp(-x))
diabetes     = np.random.binomial(1,logistic(0.6*base[:,0]-0.5),N)
hypertension = np.random.binomial(1,logistic(0.7*base[:,0]+0.3*base[:,1]-0.2),N)
obesity      = np.random.binomial(1,logistic(0.5*base[:,0]+0.4*base[:,1]-0.8),N)
ckd          = np.random.binomial(1,logistic(0.5*base[:,0]+0.6*diabetes-1.2),N)
heart_failure= np.random.binomial(1,logistic(0.4*base[:,1]+0.5*hypertension-1.5),N)
copd         = np.random.binomial(1,logistic(0.5*base[:,2]-1.0),N)
n_comorbidities = diabetes+hypertension+obesity+ckd+heart_failure+copd
readmit_logit= -2.5+0.4*heart_failure+0.3*ckd+0.2*(n_comorbidities/6)+0.3*(los_days>7)+0.2*(payers=='Medicaid')
readmitted_30= np.random.binomial(1,logistic(readmit_logit),N)
admit_type   = np.random.choice(['Emergency','Elective','Urgent','Newborn'],N,p=[0.52,0.30,0.16,0.02])
np.random.seed(21)
glucose    = np.random.normal(105,28,N).clip(50,400).round(1)
creatinine = np.random.gamma(1.6,0.6,N).clip(0.4,10).round(2)
hba1c      = np.random.normal(6.8,1.4,N).clip(4,14).round(1)

# Structured missingness
glucose[np.random.rand(N)<0.08]     = np.nan
creatinine[(payers=='Self-pay')&(np.random.rand(N)<0.50)] = np.nan
creatinine[(payers!='Self-pay')&(np.random.rand(N)<0.09)] = np.nan
charges[np.random.rand(N)<0.07]     = np.nan

start=pd.Timestamp('2019-01-01'); np.random.seed(5)
admit_dates=[start+pd.Timedelta(days=int(d))
             for d in np.random.randint(0,(pd.Timestamp('2023-12-31')-start).days,N)]

# ZIP codes (50 areas)
zip_codes = [f'{10000+np.random.randint(0,50):05d}' for _ in range(N)]
svi_by_zip = {f'{10000+i:05d}': np.random.beta(2,3) for i in range(50)}
svi_scores  = [svi_by_zip[z] for z in zip_codes]

df = pd.DataFrame({
    'patient_id'    : [f'PT-{str(i).zfill(5)}' for i in range(1,N+1)],
    'age':ages,'sex':sexes,'payer':payers,
    'drg_code':drg_codes,'drg_label':[drg_labels[d] for d in drg_codes],
    'admit_type':admit_type,'los_days':los_days,'total_charge':charges,
    'diabetes':diabetes,'hypertension':hypertension,'obesity':obesity,
    'ckd':ckd,'heart_failure':heart_failure,'copd':copd,
    'n_comorbidities':n_comorbidities,
    'glucose':glucose,'creatinine':creatinine,'hba1c':hba1c,
    'readmitted_30':readmitted_30,
    'admit_date':admit_dates,'zip':zip_codes,'svi':svi_scores,
})
df['age_group']=pd.cut(df['age'],[17,44,64,74,95],labels=['18-44','45-64','65-74','75+'])
df['admit_date']=pd.to_datetime(df['admit_date'])
df['admit_year']=df['admit_date'].dt.year
df['admit_month']=df['admit_date'].dt.to_period('M')

print(f"Capstone dataset: {df.shape}")
print(f"Readmission rate: {df['readmitted_30'].mean()*100:.1f}%")
df.head(3)


## 2. Step 1 — Data Quality Report

In [ ]:
# Quick quality audit (use ydata-profiling for full report)
print("=== DATA QUALITY AUDIT ===\n")

# Shape
print(f"Rows: {len(df):,}  |  Columns: {df.shape[1]}")

# Missing values
miss = df.isnull().sum()
miss = miss[miss>0]
print(f"\nColumns with missing data ({len(miss)}):")
for col,n in miss.items():
    print(f"  {col:20s}: {n:4d} ({n/len(df)*100:.1f}%)")

# Duplicates
print(f"\nDuplicate patient_ids: {df['patient_id'].duplicated().sum()}")

# Extreme values
print("\nExtreme value check:")
for col, lo, hi in [('age',18,95),('los_days',1,30),('glucose',50,400)]:
    out = ((df[col]<lo)|(df[col]>hi)).sum()
    print(f"  {col:15s}: {out} out-of-range values")

# Full ydata-profiling (optional — comment out if slow)
try:
    from ydata_profiling import ProfileReport
    profile = ProfileReport(df.drop(columns=['patient_id']),
                            title='Capstone Readmission EDA',minimal=True)
    profile.to_file('/tmp/mod02/capstone_profile.html')
    print("\nFull profile saved: /tmp/mod02/capstone_profile.html")
except (ImportError, ValueError, OSError):
    print("\nydata-profiling not available — manual audit above is sufficient.")


## 3. Step 2 — Table 1

In [ ]:
from scipy.stats import norm

def wilson_ci(k,n,alpha=0.05):
    z=norm.ppf(1-alpha/2); p=k/n; den=1+z**2/n
    mid=(p+z**2/(2*n))/den; hw=(z*np.sqrt(p*(1-p)/n+z**2/(4*n**2)))/den
    return round((mid-hw)*100,1),round((mid+hw)*100,1)

# Continuous: mean±SD or median[IQR]
# Categorical: N (%)
print("TABLE 1 — Baseline characteristics by 30-day readmission status")
print(f"{'':30s} {'Not readmitted':>20s} {'Readmitted':>20s} {'p-value':>10s}")
print("="*83)

g0 = df[df['readmitted_30']==0]
g1 = df[df['readmitted_30']==1]
print(f"{'N (%)':30s} {len(g0):>20,} {len(g1):>20,}")

# Continuous
for col, label in [('age','Age (years)'),('los_days','LOS (days)'),
                    ('n_comorbidities','N comorbidities')]:
    sk = skew(df[col].dropna())
    use_med = abs(sk)>0.8
    if use_med:
        s0 = f"{g0[col].median():.1f} [{g0[col].quantile(.25):.1f}–{g0[col].quantile(.75):.1f}]"
        s1 = f"{g1[col].median():.1f} [{g1[col].quantile(.25):.1f}–{g1[col].quantile(.75):.1f}]"
        stat,p = mannwhitneyu(g0[col].dropna(),g1[col].dropna(),alternative='two-sided')
    else:
        s0 = f"{g0[col].mean():.1f} ± {g0[col].std():.1f}"
        s1 = f"{g1[col].mean():.1f} ± {g1[col].std():.1f}"
        from scipy.stats import ttest_ind
        _,p = ttest_ind(g0[col].dropna(),g1[col].dropna())
    print(f"{label:30s} {s0:>20s} {s1:>20s} {p:>10.4f}")

# Categorical
for col, label in [('sex','Female sex'),('payer','Payer'),
                    ('admit_type','Admit type'),('age_group','Age group')]:
    cats = sorted(df[col].dropna().unique())
    ct = pd.crosstab(df[col],df['readmitted_30'])
    chi2,p,_,_ = chi2_contingency(ct)
    print(f"\n{label:30s}{'':20s}{'':20s} {p:>10.4f}")
    for cat in cats:
        n0=(g0[col]==cat).sum(); p0=n0/len(g0)*100
        n1=(g1[col]==cat).sum(); p1=n1/len(g1)*100
        print(f"  {str(cat):28s} {n0:>6} ({p0:4.1f}%)        {n1:>6} ({p1:4.1f}%)")


## 4. Step 3 — Temporal Trends in Readmissions

In [ ]:
monthly_read = (
    df.set_index('admit_date')
      .resample('ME')['readmitted_30']
      .agg(['sum','count'])
      .rename(columns={'sum':'n_readmit','count':'n_discharge'})
)
monthly_read['rate'] = monthly_read['n_readmit']/monthly_read['n_discharge']*100
monthly_read['roll_rate'] = monthly_read['rate'].rolling(3,center=True).mean()

fig, axes = plt.subplots(2,1,figsize=(14,7),sharex=True)
fig.suptitle('Temporal trends in hospital readmissions (2019–2023)', fontsize=12)

ax=axes[0]
ax.bar(monthly_read.index, monthly_read['n_discharge'], color='#AEC6CF', width=20, label='Discharges')
ax2=ax.twinx()
ax2.plot(monthly_read.index, monthly_read['n_readmit'], color='#D65F5F', lw=1.5, label='Readmissions')
ax.set_ylabel('N discharges'); ax2.set_ylabel('N readmissions', color='#D65F5F')
ax.set_title('Monthly volume'); ax.legend(loc='upper left'); ax2.legend(loc='upper right')

ax=axes[1]
ax.plot(monthly_read.index, monthly_read['rate'], color='#B0C4DE', lw=0.8, alpha=0.7, label='Monthly rate')
ax.plot(monthly_read.index, monthly_read['roll_rate'], color='#1F4E79', lw=2, label='3-month rolling mean')
ax.axhline(monthly_read['rate'].mean(), color='red', ls='--', lw=1.1,
           label=f"Overall mean ({monthly_read['rate'].mean():.1f}%)")
ax.set_ylabel('30-day readmission rate (%)'); ax.set_title('Readmission rate trend')
ax.legend(fontsize=9); ax.set_xlabel('Admission month')
import matplotlib.dates as mdates
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
axes[1].xaxis.set_major_locator(mdates.MonthLocator(interval=4))
plt.setp(axes[1].xaxis.get_majorticklabels(),rotation=30,ha='right')
plt.tight_layout()
plt.savefig('/tmp/mod02/capstone_temporal.png',bbox_inches='tight')
plt.show()


## 5. Step 4 — Comorbidity Analysis

In [ ]:
COMORBS = ['diabetes','hypertension','obesity','ckd','heart_failure','copd']

def phi_coeff(a,b):
    ct=pd.crosstab(a,b)
    if ct.shape!=(2,2): return 0.0
    n=ct.values.sum(); chi2=chi2_contingency(ct)[0]
    return np.sqrt(chi2/n)*np.sign(ct.iloc[1,1]*ct.iloc[0,0]-ct.iloc[0,1]*ct.iloc[1,0])

phi_m = pd.DataFrame(index=COMORBS,columns=COMORBS,dtype=float)
for c1 in COMORBS:
    for c2 in COMORBS:
        phi_m.loc[c1,c2] = phi_coeff(df[c1],df[c2])

fig,axes=plt.subplots(1,2,figsize=(15,5))
sns.heatmap(phi_m.astype(float),cmap='RdBu_r',center=0,vmin=-0.5,vmax=0.5,
            annot=True,fmt='.2f',annot_kws={'size':9},
            linewidths=0.5,ax=axes[0],cbar_kws={'label':'Phi','shrink':0.8})
axes[0].set_title('Comorbidity co-occurrence (phi)')

prev = df[COMORBS].mean()*100
readmit_rates = df.groupby(level=0)[['readmitted_30']].mean()
prev_df = pd.DataFrame({'Prevalence (%)':prev})
prev_df['Readmit rate (%)'] = [df.loc[df[c]==1,'readmitted_30'].mean()*100 for c in COMORBS]
prev_df = prev_df.sort_values('Readmit rate (%)',ascending=True)
colors_comorb = ['#D65F5F' if v>df['readmitted_30'].mean()*100 else '#4878CF'
                 for v in prev_df['Readmit rate (%)']]
axes[1].barh(prev_df.index, prev_df['Readmit rate (%)'], color=colors_comorb)
axes[1].axvline(df['readmitted_30'].mean()*100,color='black',ls='--',lw=1,
                label=f"Overall ({df['readmitted_30'].mean()*100:.1f}%)")
axes[1].set_xlabel('30-day readmission rate (%)')
axes[1].set_title('Readmission rate by comorbidity flag')
axes[1].legend(fontsize=9)
plt.tight_layout()
plt.savefig('/tmp/mod02/capstone_comorbidity.png',bbox_inches='tight')
plt.show()


## 6. Step 5 — Missingness Audit & Imputation Plan

In [ ]:
print("MISSINGNESS AUDIT\n" + "="*50)
miss_cols = ['glucose','creatinine','hba1c','total_charge']
for col in miss_cols:
    pct = df[col].isnull().mean()*100
    # Test association of missingness with payer (MAR check)
    df[f'{col}_miss'] = df[col].isnull().astype(int)
    ct = pd.crosstab(df['payer'], df[f'{col}_miss'])
    if ct.shape[1]==2:
        chi2,p,_,_ = chi2_contingency(ct)
    else:
        p = 1.0
    mechanism = 'Likely MAR (payer)' if p<0.05 else 'Possibly MCAR'
    print(f"{col:20s}: {pct:5.1f}% missing | χ²-payer p={p:.3f} → {mechanism}")

from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer
num_cols=['age','los_days','glucose','creatinine','hba1c','total_charge','n_comorbidities']
mice = IterativeImputer(random_state=SEED,max_iter=10)
df_imp = df.copy()
df_imp[num_cols] = mice.fit_transform(df[num_cols])
remaining_miss = df_imp[num_cols].isnull().sum().sum()
print(f"\nAfter MICE imputation — remaining missing: {remaining_miss}")


## 7. Step 6 — Geographic Clustering

In [ ]:
zip_agg = (
    df.groupby('zip')
      .agg(n_discharge=('patient_id','count'),
           readmit_rate=('readmitted_30',lambda x: x.mean()*100),
           svi=('svi','first'),
           mean_los=('los_days','mean'))
      .reset_index()
)
zip_agg = zip_agg[zip_agg['n_discharge']>=5]
r,p = spearmanr(zip_agg['svi'],zip_agg['readmit_rate'])
print(f"Spearman ρ (SVI vs readmit rate): {r:.3f}, p={p:.4f}")

fig,axes=plt.subplots(1,2,figsize=(13,5))
ax=axes[0]
sc=ax.scatter(zip_agg['svi'],zip_agg['readmit_rate'],
              c=zip_agg['readmit_rate'],cmap='YlOrRd',
              s=zip_agg['n_discharge']*3+20,alpha=0.75,edgecolors='white',lw=0.5)
m,b=np.polyfit(zip_agg['svi'],zip_agg['readmit_rate'],1)
xr=np.linspace(0,1,50)
ax.plot(xr,m*xr+b,'k--',lw=1.3)
ax.text(0.05,0.95,f'ρ = {r:.2f} p = {p:.3f}',transform=ax.transAxes,
        fontsize=10,va='top',bbox=dict(facecolor='white',alpha=0.8,edgecolor='none'))
plt.colorbar(sc,ax=ax,label='Readmit rate (%)',shrink=0.8)
ax.set_xlabel('Social Vulnerability Index (SVI)')
ax.set_ylabel('Readmission rate (%)')
ax.set_title('Readmission rate vs SVI by ZIP code')

ax=axes[1]
zip_agg['quintile']=pd.qcut(zip_agg['svi'],5,labels=['Q1','Q2','Q3','Q4','Q5'])
q_rates=zip_agg.groupby('quintile',observed=True)['readmit_rate'].mean()
colors_q=['#2166ac','#74add1','#fee090','#f46d43','#a50026']
bars=ax.bar(q_rates.index,q_rates.values,color=colors_q,edgecolor='white')
for bar,val in zip(bars,q_rates.values):
    ax.text(bar.get_x()+bar.get_width()/2,val+0.2,f'{val:.1f}%',ha='center',va='bottom',fontsize=9)
ax.set_xlabel('SVI quintile (Q1=least vulnerable)'); ax.set_ylabel('Mean readmission rate (%)')
ax.set_title('Readmission rate by SVI quintile')
plt.tight_layout()
plt.savefig('/tmp/mod02/capstone_geo.png',bbox_inches='tight')
plt.show()


## 8. Step 7 — Multi-Panel EDA Summary Figure

In [ ]:
fig = plt.figure(figsize=(18,12))
fig.suptitle('EDA Summary — 30-Day Readmission Cohort (N=1,000)', fontsize=15, y=1.01)
gs  = gridspec.GridSpec(3,3,figure=fig,hspace=0.45,wspace=0.35)

# Panel A — Age distribution by readmission
ax_a=fig.add_subplot(gs[0,0])
for flag,label,color in [(0,'Not readmitted','#4878CF'),(1,'Readmitted','#D65F5F')]:
    sns.kdeplot(df.loc[df['readmitted_30']==flag,'age'],ax=ax_a,
                label=label,color=color,fill=True,alpha=0.3,lw=1.5)
ax_a.set_title('A  Age distribution',fontweight='bold')
ax_a.set_xlabel('Age (years)'); ax_a.legend(fontsize=8)

# Panel B — LOS by readmission (box)
ax_b=fig.add_subplot(gs[0,1])
df_b=df[['readmitted_30','los_days']].copy()
df_b['Readmission']=['Yes' if r else 'No' for r in df_b['readmitted_30']]
sns.boxplot(data=df_b,x='Readmission',y='los_days',palette={'No':'#4878CF','Yes':'#D65F5F'},
            fliersize=2,ax=ax_b)
ax_b.set_title('B  Length of stay',fontweight='bold')
ax_b.set_ylabel('LOS (days)'); ax_b.set_xlabel('')

# Panel C — Comorbidity count distribution
ax_c=fig.add_subplot(gs[0,2])
for flag,label,color in [(0,'Not readmitted','#4878CF'),(1,'Readmitted','#D65F5F')]:
    sub=df.loc[df['readmitted_30']==flag,'n_comorbidities']
    vals,cnts=np.unique(sub,return_counts=True)
    ax_c.plot(vals,cnts/cnts.sum()*100,color=color,lw=2,marker='o',ms=5,label=label)
ax_c.set_title('C  Comorbidity burden',fontweight='bold')
ax_c.set_xlabel('N comorbidities'); ax_c.set_ylabel('% patients'); ax_c.legend(fontsize=8)

# Panel D — Readmission rate by payer
ax_d=fig.add_subplot(gs[1,0])
payer_rates=df.groupby('payer')['readmitted_30'].mean()*100
payer_rates=payer_rates.sort_values(ascending=False)
colors_p=['#D65F5F' if r>df['readmitted_30'].mean()*100 else '#4878CF' for r in payer_rates]
bars=ax_d.bar(payer_rates.index,payer_rates.values,color=colors_p,edgecolor='white')
ax_d.axhline(df['readmitted_30'].mean()*100,color='black',ls='--',lw=1)
for bar,val in zip(bars,payer_rates.values):
    ax_d.text(bar.get_x()+bar.get_width()/2,val+0.2,f'{val:.1f}%',
              ha='center',va='bottom',fontsize=8)
ax_d.set_title('D  Readmission by payer',fontweight='bold')
ax_d.set_ylabel('Readmission rate (%)'); ax_d.tick_params(axis='x',rotation=20)

# Panel E — Temporal trend
ax_e=fig.add_subplot(gs[1,1:])
ax_e.plot(monthly_read.index,monthly_read['rate'],color='#B0C4DE',lw=0.8,alpha=0.6)
ax_e.plot(monthly_read.index,monthly_read['roll_rate'],color='#1F4E79',lw=2,label='3-mo rolling mean')
ax_e.axhline(monthly_read['rate'].mean(),color='red',ls='--',lw=1,
             label=f"Mean ({monthly_read['rate'].mean():.1f}%)")
ax_e.set_title('E  Readmission rate trend 2019–2023',fontweight='bold')
ax_e.set_ylabel('Rate (%)'); ax_e.legend(fontsize=9)
import matplotlib.dates as mdates
ax_e.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax_e.xaxis.set_major_locator(mdates.MonthLocator(interval=4))
plt.setp(ax_e.xaxis.get_majorticklabels(),rotation=25,ha='right')

# Panel F — Comorbidity prevalence (readmitted vs not)
ax_f=fig.add_subplot(gs[2,0])
prev_re  = df[df['readmitted_30']==1][COMORBS].mean()*100
prev_nre = df[df['readmitted_30']==0][COMORBS].mean()*100
x=np.arange(len(COMORBS)); w=0.35
ax_f.bar(x-w/2,prev_nre,w,label='Not readmitted',color='#4878CF',alpha=0.85)
ax_f.bar(x+w/2,prev_re, w,label='Readmitted',    color='#D65F5F',alpha=0.85)
ax_f.set_xticks(x); ax_f.set_xticklabels(COMORBS,rotation=30,ha='right',fontsize=8)
ax_f.set_ylabel('Prevalence (%)'); ax_f.set_title('F  Comorbidities by outcome',fontweight='bold')
ax_f.legend(fontsize=8)

# Panel G — SVI vs readmission
ax_g=fig.add_subplot(gs[2,1])
sc_g=ax_g.scatter(zip_agg['svi'],zip_agg['readmit_rate'],
                  c=zip_agg['readmit_rate'],cmap='YlOrRd',s=50,alpha=0.7)
m2,b2=np.polyfit(zip_agg['svi'],zip_agg['readmit_rate'],1)
ax_g.plot(xr,m2*xr+b2,'k--',lw=1.3)
ax_g.set_xlabel('SVI'); ax_g.set_ylabel('Readmit rate (%)')
ax_g.set_title('G  SVI vs readmission',fontweight='bold')
plt.colorbar(sc_g,ax=ax_g,shrink=0.8)

# Panel H — Glucose by readmission
ax_h=fig.add_subplot(gs[2,2])
for flag,label,color in [(0,'Not readmitted','#4878CF'),(1,'Readmitted','#D65F5F')]:
    sns.kdeplot(df.loc[df['readmitted_30']==flag,'glucose'].dropna(),
                ax=ax_h,color=color,fill=True,alpha=0.25,lw=1.5,label=label)
ax_h.axvline(126,color='darkred',ls='--',lw=1.1,label='Diabetes threshold')
ax_h.set_title('H  Glucose by outcome',fontweight='bold')
ax_h.set_xlabel('Glucose (mg/dL)'); ax_h.legend(fontsize=8)

plt.savefig('/tmp/mod02/capstone_summary_figure.png',bbox_inches='tight',dpi=300)
plt.show()
print("Saved at 300 dpi: capstone_summary_figure.png")


## 9. Analytic Narrative Draft

> **Methods (EDA section)**
>
> We conducted a comprehensive EDA of a synthetic hospital discharge cohort (N = 1,000) spanning 2019–2023. Continuous variables were summarised as mean ± SD or median [IQR] based on Shapiro-Wilk skewness assessment; categorical variables as N (%). Group comparisons used Mann-Whitney U tests for skewed continuous outcomes and chi-square tests for categorical variables.
>
> Data quality assessment identified missing values in four variables: creatinine (high missingness in Self-pay patients, consistent with MAR mechanism), glucose (random, MCAR pattern), total charges, and HbA1c. All missing values were imputed using MICE (10 iterations, median initial strategy) with payer type included as an auxiliary variable.
>
> Comorbidity co-occurrence was assessed using pairwise phi coefficients with hierarchical Ward clustering. A 10-condition network was constructed at a phi threshold of 0.15.
>
> Geographic analysis aggregated 1,000 discharges to 50 ZIP code areas (minimum 5 discharges). Area-level readmission rates were correlated with CDC Social Vulnerability Index scores using Spearman rank correlation. A temporal decomposition of monthly admission counts used an additive seasonal model with a 12-month period.
>
> All analyses were conducted in Python 3.10 using pandas, scipy, seaborn, and sklearn. Code and datasets are available at [GitHub repository].


## 10. Final Knowledge Check

1. In the summary figure, Panel G shows ρ = +0.38 for SVI × readmission. Is this actionable? What would you do next?
2. MICE imputed creatinine for Self-pay patients. How would you validate the imputed values?
3. The temporal trend (Panel E) shows a dip in mid-2020. What would you check to determine if this is a real signal or a data artefact?
4. Heart failure and CKD have the highest readmission rates individually. What does the comorbidity heatmap tell you about patients with BOTH conditions?
5. Describe how you would extend this EDA into a formal predictive model for readmission risk.


In [ ]:
# Exercise 5 — transition to prediction
FEATURES = ['age','los_days','n_comorbidities','diabetes','hypertension',
            'ckd','heart_failure','copd']
TARGET   = 'readmitted_30'

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

X = df_imp[FEATURES]; y = df_imp[TARGET]
X_tr,X_te,y_tr,y_te = train_test_split(X,y,test_size=0.2,random_state=SEED)
rf = RandomForestClassifier(n_estimators=100,random_state=SEED,class_weight='balanced')
rf.fit(X_tr,y_tr)
auc = roc_auc_score(y_te,rf.predict_proba(X_te)[:,1])
print(f"Quick Random Forest AUC (before tuning): {auc:.3f}")
print("\nTop predictors (feature importance):")
fi = pd.Series(rf.feature_importances_,index=FEATURES).sort_values(ascending=False)
display(fi.to_frame('importance').round(4))
print("\n→ Module 04 will build on this with a full ML pipeline.")


***
## Module 02 Complete

You have worked through the full EDA toolkit for healthcare data:
- NB-01: Automated data profiling
- NB-02: Descriptive statistics and Table 1
- NB-03: Clinical distribution visualisation
- NB-04: Time-series analysis of admissions
- NB-05: Missingness analysis and MICE imputation
- NB-06: Comorbidity co-occurrence heatmap and network
- NB-07: Geospatial health disparities mapping
- NB-08: End-to-end capstone

**Next → Module 03: Statistical Inference for Health Outcomes**
